<div align="center">
  <h1>作业 3</h1>
  <div><strong>丁平尖</strong></div>
  <div>2026 年 6 月 2 日</div>
</div>

---

## 1 注意事项

1. 编程题需要打印相应的输出；
2. 将 ipynb 文件提交至 github 上，命名为：HW03-学号-姓名.ipynb。

## 2 卷积和池化层

### 2.1 理论计算题

输入一张大小为 **3 × 32 × 32**（通道数 × 高 × 宽）的彩色图像，经过一个卷积层。该卷积层包含 **16 个卷积核**，每个卷积核的大小为 **3 × 5 × 5**，并设置 **Padding = 2**、**Stride = 2**。

请计算：

1. 该卷积层输出特征图的尺寸（通道数 × 高 × 宽）。
2. 单个输出通道的一个像素值需要进行多少次乘法操作。

#### 解答

1. 输出高宽为：

   $$H_{out} = W_{out} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = 16$$

   由于卷积核个数为 16，因此输出通道数也是 16。
   所以输出特征图尺寸为 **16 × 16 × 16**。

2. 单个输出通道的一个像素值来自一个 **3 × 5 × 5** 的卷积核与对应输入区域的计算，因此需要进行：

   $$3 \times 5 \times 5 = 75$$

   即需要 **75 次乘法操作**。


### 2.2 编程题

不使用深度学习框架中的底层池化 API（如 `torch.nn.MaxPool2d`），仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持 **步幅（stride）** 和 **填充（padding）** 的二维最大池化（Max Pooling）前向传播函数。

In [3]:
import numpy as np


def max_pool2d_forward(x, kernel_size, stride=1, padding=0):
    """手动实现二维最大池化前向传播，支持 stride 和 padding。"""
    x = np.asarray(x)
    if x.ndim != 4:
        raise ValueError("Input must have shape (N, C, H, W).")

    if isinstance(kernel_size, int):
        kernel_h = kernel_w = kernel_size
    else:
        kernel_h, kernel_w = kernel_size

    if isinstance(stride, int):
        stride_h = stride_w = stride
    else:
        stride_h, stride_w = stride

    if isinstance(padding, int):
        pad_h = pad_w = padding
    else:
        pad_h, pad_w = padding

    x = x.astype(np.float32, copy=False)
    x_padded = np.pad(
        x,
        ((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)),
        mode="constant",
        constant_values=-np.inf,
    )

    batch_size, channels, padded_h, padded_w = x_padded.shape
    out_h = (padded_h - kernel_h) // stride_h + 1
    out_w = (padded_w - kernel_w) // stride_w + 1

    output = np.empty((batch_size, channels, out_h, out_w), dtype=x_padded.dtype)

    for batch_index in range(batch_size):
        for channel_index in range(channels):
            for out_i in range(out_h):
                h_start = out_i * stride_h
                h_end = h_start + kernel_h
                for out_j in range(out_w):
                    w_start = out_j * stride_w
                    w_end = w_start + kernel_w
                    window = x_padded[batch_index, channel_index, h_start:h_end, w_start:w_end]
                    output[batch_index, channel_index, out_i, out_j] = np.max(window)

    return output


# 简单验证
x_demo = np.array(
    [[[[1, 2, 3, 4],
       [5, 6, 7, 8],
       [9, 10, 11, 12],
       [13, 14, 15, 16]]]],
    dtype=np.float32,
)

pooled_demo = max_pool2d_forward(x_demo, kernel_size=2, stride=2, padding=0)
print("output shape:", pooled_demo.shape)
print(pooled_demo)


output shape: (1, 1, 2, 2)
[[[[ 6.  8.]
   [14. 16.]]]]


## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

在 VGG 网络中，作者常用多个 **3 × 3** 卷积核级联来代替较大的卷积核（如 **5 × 5** 或 **7 × 7**）。设输入和输出特征图的通道数均为 **C**。

请计算：

1. 一个 **5 × 5** 卷积层（不带偏置）的参数量。
2. 两个串联的 **3 × 3** 卷积层（不带偏置，且两层通道数都为 **C**）的总参数量。

#### 解答

1. 一个 5 × 5 卷积层的参数量为：

   $$C \times C \times 5 \times 5 = 25C^2$$

2. 两个串联的 3 × 3 卷积层的参数量为：

   $$C \times C \times 3 \times 3 + C \times C \times 3 \times 3 = 18C^2$$

   因此，两个 3 × 3 卷积层的总参数量更少，同时也与一个 5 × 5 卷积层相当。



### 3.2 编程题

NiN 网络的核心创新是引入“1 × 1 卷积”组成的 NiN 块来代替传统全连接层，以减少参数量。请使用 PyTorch 的 `torch.nn.Sequential` 定义一个标准的 NiN 块（NiN Block）。

要求：NiN 块接收输入通道数 `in_channels` 和输出通道数 `out_channels`，它由一个普通卷积层（指定窗口大小 `kernel_size`、步幅 `stride`、填充 `padding`）以及两个随后的 **1 × 1 卷积层** 级联组成。每层卷积后都需要紧跟一个 ReLU 激活层。

In [8]:
import torch
import torch.nn as nn


def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


# 构造一个 NiN 块
net = nin_block(3, 96, kernel_size=11, stride=4, padding=0)
print(net)


Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU()
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)


## 4 Inception、批量归一化和残差网络

### 4.1 理论计算题

在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_1=2$，$x_2=4$，$x_3=6$，$x_4=8$。假设当前批量归一化层学到的缩放参数 $\gamma=2$，平移参数 $\beta=1$，常数 $\epsilon=0$。

请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 $y_1,y_2,y_3,y_4$。

#### 解答

先计算均值与方差：

$$\mu = \frac{2+4+6+8}{4} = 5$$

$$\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = 5$$

标准化后：

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}$$

再进行缩放和平移：

$$y_i = \gamma \hat{x}_i + \beta = 2\cdot\frac{x_i - 5}{\sqrt{5}} + 1$$

因此：

- $y_1 = 1 - \frac{6}{\sqrt{5}}$
- $y_2 = 1 - \frac{2}{\sqrt{5}}$
- $y_3 = 1 + \frac{2}{\sqrt{5}}$
- $y_4 = 1 + \frac{6}{\sqrt{5}}$



### 4.2 编程题

残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的梯度消失问题。请用 PyTorch 自定义一个残差块类 `Residual`。

要求：该块包含两个具有相同输出通道数的 $3 \times 3$ 卷积层，每个卷积层后跟一个批量归一化层。如果 `use_1x1conv=True`，则需要对输入应用一个 $1 \times 1$ 的卷积层来调整输入的通道数和形状，以便它能和第二层卷积的输出进行按元素相加（$f(x)+x$）。

In [10]:
import torch
from torch import nn


class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
        self.relu = nn.ReLU()

    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        y = y + x
        return self.relu(y)


# 简单验证
res_block = Residual(3, 6, use_1x1conv=True, stride=2)
print(res_block)


Residual(
  (conv1): Conv2d(3, 6, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(6, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(3, 6, kernel_size=(1, 1), stride=(2, 2))
  (relu): ReLU()
)


## 5 图像增广、微调和样式迁移

### 5.1 理论计算题

在微调（Fine-tuning）任务中，我们通常会在一个大型源数据集（如 ImageNet）上预训练好的网络模型基础上，去适应一个新的目标数据集。

请回答以下问题：

1. 为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出层”设置较大的学习率？
2. 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？

#### 解答

1. 底层特征提取层通常已经学到了较通用的视觉特征，如边缘、纹理和局部形状，这些特征在新任务中往往仍然有用，因此一般不需要大幅调整。对它们使用较小学习率或直接冻结参数，可以保留预训练知识并减少训练开销。顶层输出层则是随机初始化的，需要更快地适应新任务，所以通常设置较大的学习率。
2. 如果目标数据集很小且与源数据集相似，通常应优先冻结大部分底层网络，只微调最后几层或输出层，并配合较小学习率、早停等策略，以降低过拟合风险。



### 5.2 编程题

图像增广能有效增强模型的泛化能力。请利用 `torchvision.transforms` 模块创建一个组合图像增广管道（Pipeline）。

要求：

1. 随机对图像进行裁剪，使其面积比例在 0.08 到 1.0 之间，并将裁剪后的图像缩放到 224 × 224 像素。
2. 拥有 50% 的概率对图像进行水平翻转。
3. 随机改变图像的亮度（Brightness）、对比度（Contrast）和饱和度（Saturation），变化范围设为 0.5。
4. 最终将图像转换为 PyTorch 张量（Tensor）。

In [12]:
from torchvision import transforms


train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor(),
])

print(train_transforms)


Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重合程度。已知图像中两个边界框（以 `[左上角 x, 左上角 y, 右下角 x, 右下角 y]` 格式表示）：

1. 真实框（Ground Truth）A = [10, 10, 50, 50]
2. 预测框（Prediction Box）B = [30, 30, 70, 70]

请计算边界框 A 和边界框 B 之间的 IoU 准确值。

#### 解答

两个边界框的交集区域为：

- 左上角坐标为 `(30, 30)`
- 右下角坐标为 `(50, 50)`

因此交集宽高都为 `20`，交集面积为：

$$S_{inter} = 20 \times 20 = 400$$

两个边界框各自的面积为：

$$S_A = 40 \times 40 = 1600$$

$$S_B = 40 \times 40 = 1600$$

并集面积为：

$$S_{union} = S_A + S_B - S_{inter} = 1600 + 1600 - 400 = 2800$$

所以 IoU 为：

$$IoU = \frac{S_{inter}}{S_{union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$



### 6.2 编程题

在计算机视觉训练技巧中，标签平滑（Label Smoothing）通过防止模型过于自信地预测某些类别来提高泛化性。标准交叉熵使用独热编码（One-hot），若设置平滑因子 `ε=0.1`，则对于 `K` 分类问题，真实标签对应的目标概率从 `1` 变为 `1 - ε`，其余错误类别的概率从 `0` 变为 `ε / (K - 1)`。

请实现一个计算标签平滑后交叉熵损失的函数。

In [14]:
import torch
import torch.nn.functional as F


def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """计算带标签平滑的交叉熵损失。"""
    if logits.ndim != 2:
        raise ValueError("logits must have shape (N, K).")
    if targets.ndim != 1:
        raise ValueError("targets must have shape (N,).")

    num_classes = logits.shape[1]
    log_probs = F.log_softmax(logits, dim=1)
    one_hot = F.one_hot(targets, num_classes=num_classes).to(logits.dtype)
    smooth_targets = one_hot * (1 - epsilon) + (1 - one_hot) * (epsilon / (num_classes - 1))
    loss = -(smooth_targets * log_probs).sum(dim=1).mean()
    return loss


# 简单验证
logits_demo = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.2, 0.3]])
targets_demo = torch.tensor([0, 1])
loss_demo = label_smoothing_cross_entropy(logits_demo, targets_demo, epsilon=0.1)
print(loss_demo)


tensor(0.5599)
